# GRAND Algorithm for BB84 Error Mitigation

---

## Overview

**GRAND** (Guessing Random Additive Noise Decoding) is an information-theoretic error correction algorithm that:

1. **Receives** a noisy key (Bob's sifted key with potential errors)
2. **Guesses** random error patterns with increasing Hamming weight
3. **Adds** each guess to the received key (XOR operation)
4. **Validates** each candidate against a criterion (oracle, syndrome, or parity)
5. **Returns** the first valid candidate found

### Key Advantages

- **Model-agnostic**: works with any code, not tied to specific structure
- **Capacity-achieving**: asymptotically optimal for random errors
- **Simple to implement**: no syndrome computation needed (can use oracle)
- **Practical**: effective for low-error regimes typical in QKD

### Trade-offs

- **Computational cost**: grows with error weight (mitigated by adaptive search)
- **No theoretical guarantee** for large error correction capability
- **Oracle-based validation** requires knowing the correct key

---

## Setup & Imports

In [1]:
import importlib
import sys
import pathlib
import os

# Clear cached modules
for mod in list(sys.modules.keys()):
    if "bb84" in mod:
        sys.modules.pop(mod)
importlib.invalidate_caches()

# Add parent to path
project_root = pathlib.Path().resolve().parent
sys.path.insert(0, str(project_root))

# Imports
import numpy as np
import random
from typing import List, Tuple, Dict, Optional
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from bb84_config import SimulationConfig, QBERResult, SimulationResult
from bb84_runner import run_simulation
from bb84_core import estimate_qber
from bb84_grand import (
    GRANDDecoder,
    GRANDDecodeResult,
    correct_sifted_key_with_grand,
    analyze_grand_performance,
    introduce_errors,
)

print("[✓] All imports successful")

[✓] All imports successful


---

## Experiment 1: Simple Demonstration with Controlled Errors

Start with a small example to illustrate GRAND functionality.

In [ ]:
# Create a simple test key
np.random.seed(42)
alice_key_small = [int(b) for b in np.random.randint(0, 2, 15)]
print(f"Alice's key:    {alice_key_small}")
print(f"Key length:     {len(alice_key_small)}")

# Introduce errors (20% error rate)
bob_key_noisy, num_errors = introduce_errors(alice_key_small, error_rate=0.2, seed=99)
print(f"\nBob's noisy key: {bob_key_noisy}")
print(f"Errors introduced: {num_errors}")

# Compute error positions
error_positions = [i for i in range(len(alice_key_small)) 
                  if alice_key_small[i] != bob_key_noisy[i]]
print(f"Error positions: {error_positions}")

Alice's key:    [0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1]
Key length:     15

Bob's noisy key: [0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1]
Errors introduced: 3
Error positions: [7, 11, 12]


: 

In [ ]:
# Run GRAND decoding
corrected, result = correct_sifted_key_with_grand(
    bob_key_noisy,
    alice_sifted_key=alice_key_small,
    validation_mode="oracle",
    estimated_qber=0.15,  # ~20% errors expected
)

print("\n" + "="*60)
print("GRAND Decoding Result")
print("="*60)
print(f"Alice's key:        {alice_key_small}")
print(f"Bob's key (noisy):  {bob_key_noisy}")
print(f"Corrected key:      {corrected}")
print(f"\nDecoding result: {'SUCCESS ✓' if result.success else 'FAILURE ✗'}")

if result.success:
    print(f"  Guesses tried:        {result.guesses_tried}")
    print(f"  Error pattern weight: {result.error_pattern_weight}")
    print(f"  Validation method:    {result.validation_method}")
    
    # Verify correction
    if corrected == alice_key_small:
        print(f"\n  ✓ Key fully corrected!")
    else:
        residual_errors = sum(1 for a, c in zip(alice_key_small, corrected) if a != c)
        print(f"\n  ✗ {residual_errors} errors remain")
else:
    print(f"  Max guesses attempted: {result.guesses_tried}")
    print(f"  Estimated BER: {result.estimated_ber:.3f}")

---

## Experiment 2: GRAND vs. Error Rate

Test how GRAND performance varies with increasing error rates.

In [ ]:
# Test across a range of error rates
key_length = 100  # Use longer key for statistical significance
error_rates = np.linspace(0.01, 0.15, 8)
num_trials_per_rate = 20

results_by_rate = {}

print("Testing GRAND across error rates...\n")
print(f"{'Error Rate':<12} {'Success Rate':<15} {'Avg Guesses':<15} {'Avg Weight':<12}")
print("-" * 54)

np.random.seed(42)

for error_rate in error_rates:
    trial_results = []
    
    for trial in range(num_trials_per_rate):
        # Generate random Alice key
        alice_key = [int(b) for b in np.random.randint(0, 2, key_length)]
        
        # Introduce errors
        bob_key_noisy, _ = introduce_errors(
            alice_key,
            error_rate=error_rate,
            seed=42 + trial,
        )
        
        # Attempt GRAND correction
        corrected, result = correct_sifted_key_with_grand(
            bob_key_noisy,
            alice_sifted_key=alice_key,
            validation_mode="oracle",
            estimated_qber=error_rate,
        )
        
        trial_results.append(result)
    
    # Analyze results for this rate
    metrics = analyze_grand_performance(trial_results)
    results_by_rate[error_rate] = (metrics, trial_results)
    
    success_rate = metrics.get("success_rate", 0.0)
    avg_guesses = metrics.get("avg_guesses_per_success", 0.0)
    avg_weight = metrics.get("avg_error_weight", 0.0)
    
    status = "✓" if success_rate > 0.9 else "" if success_rate > 0.5 else "✗"
    print(f"{error_rate:>6.2%}      {success_rate:>6.1%}         {avg_guesses:>8.1f}       {avg_weight:>6.2f}  {status}")

print("\nDone!")

Plot the results:

In [ ]:
# Extract metrics for plotting
error_rates_list = sorted(results_by_rate.keys())
success_rates = [results_by_rate[r][0]['success_rate'] for r in error_rates_list]
avg_guesses_list = [results_by_rate[r][0]['avg_guesses_per_success'] for r in error_rates_list]
avg_weights = [results_by_rate[r][0]['avg_error_weight'] for r in error_rates_list]

# Create figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Success Rate
ax = axes[0]
ax.plot(error_rates_list, success_rates, 'o-', linewidth=2, markersize=8, color='green')
ax.axhline(y=0.95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')
ax.set_xlabel('Error Rate (QBER)', fontsize=11)
ax.set_ylabel('Success Rate', fontsize=11)
ax.set_title('GRAND: Success Rate vs Error Rate', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
ax.legend()

# Plot 2: Average Guesses
ax = axes[1]
ax.semilogy(error_rates_list, avg_guesses_list, 's-', linewidth=2, markersize=8, color='blue')
ax.set_xlabel('Error Rate (QBER)', fontsize=11)
ax.set_ylabel('Avg Guesses (log scale)', fontsize=11)
ax.set_title('GRAND: Computational Cost vs Error Rate', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

# Plot 3: Average Error Weight Detected
ax = axes[2]
expected_weights = [int(key_length * r) for r in error_rates_list]
ax.plot(error_rates_list, avg_weights, 'D-', linewidth=2, markersize=8, 
        color='red', label='GRAND detected')
ax.plot(error_rates_list, expected_weights, '--', linewidth=2, color='orange', 
        label='Expected weight')
ax.set_xlabel('Error Rate (QBER)', fontsize=11)
ax.set_ylabel('Hamming Weight', fontsize=11)
ax.set_title('GRAND: Error Pattern Weight', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.savefig('../images/grand_performance_vs_error_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print("[✓] Plot saved to ../images/grand_performance_vs_error_rate.png")

---

## Experiment 3: Integration with BB84 QKD Simulator

Apply GRAND to actual BB84 noisy channel outputs.

In [ ]:
# Configure a BB84 simulation with noise
config = SimulationConfig(
    n_qubits=500,
    seed=42,
    label="BB84 with Noise",
    noise_model="depolarizing",  # Phase 3 noise model
    depolar_prob=0.05,
    eve_present=False,
    sample_fraction=0.15,
)

print(f"Running BB84 simulation: {config.n_qubits} qubits...")
sim_result = run_simulation(config)

print(f"\nSimulation Results:")
print(f"  Sifted key length: {len(sim_result.alice_sifted_key)}")
print(f"  Estimated QBER: {sim_result.qber_result.qber:.4f}")
print(f"  Detected errors in sample: {sim_result.qber_result.errors}")

In [ ]:
# Apply GRAND to correct Bob's key
print("Applying GRAND error correction...\n")

corrected_key, grand_result = correct_sifted_key_with_grand(
    sim_result.bob_sifted_key,
    alice_sifted_key=sim_result.alice_sifted_key,
    validation_mode="oracle",
    estimated_qber=sim_result.qber_result.qber,
)

# Compute residual error
if grand_result.success:
    residual_errors = sum(1 for a, c in zip(sim_result.alice_sifted_key, corrected_key)
                          if a != c)
    print(f"GRAND Result: SUCCESS ✓")
    print(f"  Guesses needed: {grand_result.guesses_tried}")
    print(f"  Error pattern weight: {grand_result.error_pattern_weight}")
    print(f"  Residual errors: {residual_errors}")
else:
    print(f"GRAND Result: FAILURE ✗")
    print(f"  Max guesses tried: {grand_result.guesses_tried}")

# Compute bit-flip rate comparison
original_errors = sum(1 for a, b in zip(sim_result.alice_sifted_key, sim_result.bob_sifted_key)
                       if a != b)
original_ber = original_errors / len(sim_result.alice_sifted_key) if sim_result.alice_sifted_key else 0

if grand_result.success:
    corrected_errors = sum(1 for a, c in zip(sim_result.alice_sifted_key, corrected_key)
                            if a != c)
    corrected_ber = corrected_errors / len(sim_result.alice_sifted_key) if sim_result.alice_sifted_key else 0
    
    print(f"\nBit Error Rate:")
    print(f"  Before GRAND: {original_ber:.4f} ({original_errors} errors)")
    print(f"  After GRAND:  {corrected_ber:.4f} ({corrected_errors} errors)")
    print(f"  Improvement:  {100*(original_ber - corrected_ber)/original_ber:.1f}% reduction")

---

## Experiment 4: Validation Mode Comparison

Compare different validation strategies: oracle, syndrome, and parity.

In [ ]:
# Generate a test key with known errors
test_key_length = 80
alice_test = [int(b) for b in np.random.randint(0, 2, test_key_length)]
bob_test, actual_errors = introduce_errors(alice_test, error_rate=0.08, seed=123)

print(f"Test key: {test_key_length} bits")
print(f"Actual errors introduced: {actual_errors}")
print(f"BER: {actual_errors/test_key_length:.3f}\n")

# Test each validation mode
validation_modes = ["oracle", "parity"]
results_by_mode = {}

for mode in validation_modes:
    print(f"Testing validation mode: '{mode}'")
    
    corrected, result = correct_sifted_key_with_grand(
        bob_test,
        alice_sifted_key=alice_test if mode == "oracle" else None,
        validation_mode=mode,
        estimated_qber=0.08,
    )
    
    results_by_mode[mode] = result
    
    print(f"  Result: {'SUCCESS ✓' if result.success else 'FAILURE ✗'}")
    print(f"  Guesses tried: {result.guesses_tried}")
    if result.success:
        print(f"  Error weight: {result.error_pattern_weight}")
    print()

---

## Experiment 5: GRAND Scalability

Test how GRAND scales with key length.

In [ ]:
# Test across different key lengths
key_lengths = [50, 100, 200, 400, 800]
fixed_error_rate = 0.05
trials_per_length = 15

scalability_results = {}

print("Testing GRAND scalability...\n")
print(f"{'Key Length':<12} {'Success Rate':<15} {'Avg Guesses':<15} {'Avg Time':<12}")
print("-" * 54)

import time

np.random.seed(42)

for key_length in key_lengths:
    trial_results = []
    times = []
    
    for trial in range(trials_per_length):
        # Generate random keys
        alice_key = [int(b) for b in np.random.randint(0, 2, key_length)]
        bob_key_noisy, _ = introduce_errors(
            alice_key,
            error_rate=fixed_error_rate,
            seed=42 + trial,
        )
        
        # Time GRAND execution
        start = time.perf_counter()
        corrected, result = correct_sifted_key_with_grand(
            bob_key_noisy,
            alice_sifted_key=alice_key,
            validation_mode="oracle",
            estimated_qber=fixed_error_rate,
        )
        elapsed = time.perf_counter() - start
        
        trial_results.append(result)
        times.append(elapsed)
    
    metrics = analyze_grand_performance(trial_results)
    avg_time = np.mean(times)
    scalability_results[key_length] = (metrics, trial_results)
    
    success_rate = metrics.get("success_rate", 0.0)
    avg_guesses = metrics.get("avg_guesses_per_success", 0.0)
    
    print(f"{key_length:>6} bits    {success_rate:>6.1%}         {avg_guesses:>8.1f}       {avg_time*1000:>8.2f} ms")

print("\nDone!")

In [ ]:
# Plot scalability
key_lengths_list = sorted(scalability_results.keys())
success_rates_scale = [scalability_results[k][0]['success_rate'] for k in key_lengths_list]
avg_guesses_scale = [scalability_results[k][0]['avg_guesses_per_success'] for k in key_lengths_list]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Success rate vs key length
ax = axes[0]
ax.plot(key_lengths_list, success_rates_scale, 'o-', linewidth=2, markersize=8, color='green')
ax.axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Key Length (bits)', fontsize=11)
ax.set_ylabel('Success Rate', fontsize=11)
ax.set_title('GRAND: Scalability with Key Length\n(fixed 5% error rate)', 
            fontsize=12, fontweight='bold')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Computational cost vs key length
ax = axes[1]
ax.loglog(key_lengths_list, avg_guesses_scale, 's-', linewidth=2, markersize=8, color='blue')
ax.set_xlabel('Key Length (bits)', fontsize=11)
ax.set_ylabel('Avg Guesses (log scale)', fontsize=11)
ax.set_title('GRAND: Computational Cost Scaling', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('../images/grand_scalability.png', dpi=150, bbox_inches='tight')
plt.show()

print("[✓] Plot saved to ../images/grand_scalability.png")

---

## Summary & Recommendations

### When to Use GRAND for BB84:

1. **Low error regimes** (QBER < 15%): GRAND is highly effective
2. **Model-agnostic** error correction needed: works without code structure assumptions
3. **Oracle available**: when Alice's key is known (practical in QKD post-processing)
4. **Computational budget available**: searches scale with error weight, not exponentially

### Limitations:

1. **High QBER regimes**: (>20%) GRAND cost becomes prohibitive
2. **No syndrome-based speedup**: oracle validation is simple but requires known key
3. **Security consideration**: GRAND reveals information during decoding (consider noise)

### Integration into Full BB84 Pipeline:

```python
# After basis reconciliation (sifting) and QBER estimation:
qber_estimated = estimate_qber(...)

# Apply GRAND
final_key, result = correct_sifted_key_with_grand(
    bob_sifted_key,
    alice_sifted_key=alice_sifted_key,
    validation_mode="oracle",
    estimated_qber=qber_estimated.qber
)

# If GRAND fails, implement information reconciliation (next phase)
if not result.success:
    # Use classical error correction (e.g., Hamming, BCH, LDPC)
    pass
```

### Future Enhancements:

- Implement syndrome-based validation for faster decoding
- Add multi-threaded candidate testing
- Integrate classical error correction codes (Hamming, BCH, LDPC)
- Combine GRAND with privacy amplification